# Imports

In [ ]:
import torch
import os
import sys

import numpy as np
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from craft.craft_torch import Craft, torch_to_numpy
import seaborn as sb


import matplotlib.pyplot as plt
import random as rd
import pickle as pkl
import scipy.special as sc
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
sys.path.append(os.getcwd())
result_folder = ""

# Results

## Bias alignment results
Results for part 4.1 of the paper and appendix B and C.2

### CMNIST bias alignment
Appendix B

In [ ]:
def get_align_res(model_type, exp_type, concept_range, patch_range, exp_id_range, overwrite = False):
    res_path = f"{result_folder}/models/{model_type}/results_{model_type}{exp_type}.pkl"
    if os.path.exists(res_path) and not overwrite:
        with open(res_path, "rb") as f:
            return pkl.load(f)
    res = []
    for patch_size in patch_range:
        res.append([])
        for concept_id in concept_range:
            res[-1].append([])
            for exp_id in exp_id_range:
                with open(f"{result_folder}/models/{model_type}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_size}.pkl", "rb") as f:
                    res[-1][-1] += [np.abs(el).max() for el in pkl.load(f)["bias_alignment_values"]]
    with open(res_path, "wb") as f:
        pkl.dump(res, f)
    return res


resbb = get_align_res("CMNISTb", "b", range(5, 16), [4, 6, 8, 10], range(10))
print("Done bb")
resbu = get_align_res("CMNISTb", "u", range(5, 16), [4, 6, 8, 10], range(10))
print("Done bu")
resbob = get_align_res("CMNISTb", "ob", range(5, 16), [4, 6, 8, 10], range(10))
print("Done bob")
resub = get_align_res("CMNISTu", "b", range(5, 16), [4, 6, 8, 10], range(10))
print("Done ub")
resuu = get_align_res("CMNISTu", "u", range(5, 16), [4, 6, 8, 10], range(10))
print("Done uu")

In [ ]:
from scipy.stats import sem
fig, axes = plt.subplots(1, 6, figsize=(35*2,3.8), width_ratios=[0.33, 0.33, 0.33, 0.33, 0.33, 0.02])
vmin = 0.5
vmax = 0.8


alignment_resbb = [[sum(el)/len(el) for el in row] for row in resbb]
variance = [[np.array(el).std() for el in row] for row in resbb]
ax = axes[0]
sb.heatmap([[el for el in row] for row in alignment_resbb], ax=ax, vmin=vmin, vmax=vmax, annot=variance, xticklabels=range(5, 16), yticklabels=[4, 6, 8, 10], cbar=False)
ax.set_xlabel("Amount of concept")
ax.set_ylabel("Patch size")
ax.set_title("a) Biased model, biased audit")

ax=axes[1]
alignment_resbu = [[sum(el)/len(el) for el in row] for row in resbu]
variance = [[np.array(el).std() for el in row] for row in resbu]
sb.heatmap([[el for el in row] for row in alignment_resbu], ax=ax, vmin=vmin, vmax=vmax, annot=variance, xticklabels=range(5, 16), yticklabels=[4, 6, 8, 10], cbar_ax=axes[5])
ax.set_xlabel("Amount of concept")
# ax.set_ylabel("Patch size")
ax.set_title("b) Biased model, unbiased audit")

ax=axes[2]
alignment_resbob = [[sum(el)/len(el) for el in row] for row in resbob]
variance = [[np.array(el).std() for el in row] for row in resbob]
sb.heatmap([[el for el in row] for row in alignment_resbob], ax=ax, vmin=vmin, vmax=vmax, annot=variance, xticklabels=range(5, 16), yticklabels=[4, 6, 8, 10], cbar=False)
ax.set_xlabel("Amount of concept")
# ax.set_ylabel("Patch size")
ax.set_title("c) Biased model, differently biased audit")

ax=axes[3]
alignment_resub = [[sum(el)/len(el) for el in row] for row in resub]
variance = [[np.array(el).std() for el in row] for row in resub]
sb.heatmap([[el for el in row] for row in alignment_resub], ax=ax, vmin=vmin, vmax=vmax, annot=variance, xticklabels=range(5, 16), yticklabels=[4, 6, 8, 10], cbar=False)
ax.set_xlabel("Amount of concept")
# ax.set_ylabel("Patch size")
ax.set_title("d) Unbiased model, biased audit")

ax=axes[4]
alignment_resuu = [[sum(el)/len(el) for el in row] for row in resuu]
variance = [[np.array(el).std() for el in row] for row in resuu]
sb.heatmap([[el for el in row] for row in alignment_resuu], ax=ax, vmin=vmin, vmax=vmax, annot=variance, xticklabels=range(5, 16), yticklabels=[4, 6, 8, 10], cbar=False)
ax.set_xlabel("Amount of concept")
# ax.set_ylabel("Patch size")
ax.set_title("e) Unbiased model, unbiased audit")

### Waterbirds bias alignment
Part 4.1

In [ ]:
def get_waterbirds_highest_bias_alignment(exp_name,  exp_id, concept_id, patch_id):
    concept_amounts = [5, 10, 20]
    patch_sizes = [25, 50, 75]
    number_of_concepts = concept_id
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_b_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_res = pkl.load(f)
    res =  []
    for studied_bias in range(2):
        crops_u = concept_res["concept_parameters"][studied_bias]["crops_u"]
        crops = concept_res["concept_parameters"][studied_bias]["crops"]
        masks = concept_res["concept_parameters"][studied_bias]["crop_masks"]
        tmp = 1
        for c_id in range(number_of_concepts):
            best_crops_ids = np.argsort(crops_u[:, c_id])[::-1][:100]
            best_masks = masks[best_crops_ids]
            alignment = (((best_masks > 0).mean()))
            if alignment < tmp:
                tmp = alignment
        
        res.append(1-tmp)
    return res


def get_waterbirds_align_res(model_type, exp_type, concept_range, patch_range, exp_id_range, overwrite = False):
    res_path = f"{result_folder}/models/{model_type}/results_{model_type}{exp_type}.pkl"
    if os.path.exists(res_path) and not overwrite:
        with open(res_path, "rb") as f:
            return pkl.load(f)
    res = []
    for patch_size in patch_range:
        res.append([])
        for concept_id in concept_range:
            res[-1].append([])
            for exp_id in exp_id_range:
                res[-1][-1] += get_waterbirds_highest_bias_alignment(model_type, exp_id, concept_id, patch_size)
    with open(res_path, "wb") as f:
        pkl.dump(res, f)
    return res

if not os.path.exists(f"{result_folder}/models/Waterbirds18/results_Waterbirds18b.pkl"):
    res18 = get_waterbirds_align_res(model_type="Waterbirds18", exp_type="b", concept_range=[5, 10, 15], patch_range=[25, 50, 75], exp_id_range=range(10), overwrite=True)
else:
    with open(f"{result_folder}/models/Waterbirds18/results_Waterbirds18b.pkl", "rb") as f:
        res18 = pkl.load(f)

In [ ]:
alignment_res18 = [[sum(el)/len(el) for el in row] for row in res18]
variance = [[sem(el) for el in row] for row in res18]
fig = sb.heatmap([[el for el in row] for row in alignment_res18], vmin=0.8, vmax=1, annot=True, xticklabels=[5, 10, 20], yticklabels=[25, 50, 75], cbar=True)
fig.set_xlabel("Amount of concept")
fig.set_ylabel("Patch size")

### CMNIST bias estimator alignment

#### Check for experiment
Sanity check of how the concepts are modifed by the gradient (both descend and ascend)

In [ ]:
def get_cmnist_bias_alignment(exp_name, studied_bias,  exp_id, concept_id, patch_id, exp_type="b", make_fig=True):
    with open(f"{result_folder}/models/{exp_name}/bias_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        bias_res = pkl.load(f)
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_res = pkl.load(f)
    W = concept_res["concept_parameters"][studied_bias]["W"]
    big_res = []
    for label in bias_res[studied_bias]:
        truc = bias_res[studied_bias][label][2]
        big_res.append(cos_sim([truc.mean(axis=0)], W)[0])
    big_res.append(sum(big_res)/len(big_res))
    if make_fig:sb.heatmap(big_res, center=0, annot=False, cbar=True, yticklabels=list(range(10)) + ["Avg"])
    return big_res[-1]

def show_freq(exp_name, exp_id, studied_bias, concept_id, patch_id, exp_type="b", make_fig = True):
    amount_of_concept = concept_id
    patch_size = patch_id
    if make_fig:
        fig, axes = plt.subplots(5, 2, figsize=(50,30))
    steps_amount = [1000, 5000, 10000, 20000] #steps_amount
    fontsize = 32
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_res = pkl.load(f)

    base_freq = []
    ascent_freq = []
    descent_freq = []
    for step_amount in steps_amount:
        if "false_n" in concept_res["concept_results"][step_amount][studied_bias]:
            results = concept_res["concept_results"][step_amount][studied_bias]["false_n"]
            appearance = len(results["concept_base"])
            base_freq.append(((results["concept_base"] > 0).sum(axis=0))/appearance)
            ascent_freq.append((results["concept_ascent"] > 0).sum(axis=0)/appearance)
            descent_freq.append((results["concept_descent"] > 0).sum(axis=0)/appearance)
        else:
            base_freq.append(np.array([0 for i in range(amount_of_concept)]))
            ascent_freq.append(np.array([0 for i in range(amount_of_concept)]))
            descent_freq.append(np.array([0 for i in range(amount_of_concept)]))
    base_freq = np.stack(base_freq)
    ascent_freq = np.stack(ascent_freq)
    descent_freq = np.stack(descent_freq)
    if make_fig:
        sb.heatmap(ascent_freq, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[0][0])
        axes[0][0].set_title("Ascent frequency", {"fontsize":fontsize})

        sb.heatmap(base_freq, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[1][0])
        axes[1][0].set_title("Base frequency", {"fontsize":fontsize})

        sb.heatmap(descent_freq, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[2][0])
        axes[2][0].set_title("Descent frequency", {"fontsize":fontsize})

    false_n_res = descent_freq - base_freq
    false_n_res[false_n_res < 0] = 0
    # false_n_res[base_freq < 1] /= 1 - base_freq[base_freq < 1]
    if make_fig:
        sb.heatmap(false_n_res, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[3][0])
        axes[3][0].set_title("False negatives results", {"fontsize":fontsize})

    base_freq = []
    ascent_freq = []
    descent_freq = []
    for step_amount in steps_amount:
        if "false_p" in concept_res["concept_results"][step_amount][studied_bias]:
            results = concept_res["concept_results"][step_amount][studied_bias]["false_p"]
            appearance = len(results["concept_base"])
            base_freq.append(((results["concept_base"] > 0).sum(axis=0))/appearance)
            ascent_freq.append((results["concept_ascent"] > 0).sum(axis=0)/appearance)
            descent_freq.append((results["concept_descent"] > 0).sum(axis=0)/appearance)
        else: 
            base_freq.append(np.array([0 for i in range(amount_of_concept)]))
            ascent_freq.append(np.array([0 for i in range(amount_of_concept)]))
            descent_freq.append(np.array([0 for i in range(amount_of_concept)]))
    base_freq = np.stack(base_freq)
    ascent_freq = np.stack(ascent_freq)
    descent_freq = np.stack(descent_freq)
    if make_fig:
        sb.heatmap(ascent_freq, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[0][1])
        axes[0][1].set_title("Ascent frequency", {"fontsize":fontsize})

        sb.heatmap(base_freq, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[1][1])
        axes[1][1].set_title("Base frequency", {"fontsize":fontsize})

        sb.heatmap(descent_freq, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[2][1])
        axes[2][1].set_title("Descent frequency", {"fontsize":fontsize})

    false_p_res = (base_freq - descent_freq)
    false_p_res[false_p_res < 0] = 0
    # false_p_res[base_freq > 0] /= base_freq[base_freq > 0]
    if make_fig:
        sb.heatmap(false_p_res, vmin=0, vmax=1, annot=True, yticklabels=steps_amount, ax=axes[3][1])
        axes[3][1].set_title("False negatives results", {"fontsize":fontsize})

    final_res = (false_p_res + false_n_res) /2
    if make_fig:
        sb.heatmap(final_res, center=0, annot=True, yticklabels=steps_amount, ax=axes[4][1])
        axes[4][1].set_title("Final bias estimator", {"fontsize":fontsize})
    
    bias_align = [[el for el in concept_res["bias_alignment_values"][studied_bias]], concept_res["bias_alignment_values"][studied_bias]]
    if make_fig:
        sb.heatmap(bias_align, vmin=0, vmax=1, annot=True, ax=axes[4][0])
        ax=axes[4][0].set_title("Bias alignment values", {"fontsize":fontsize})

    return final_res
# get_waterbirds_concept_relevance(exp_name="Waterbirds18", exp_id=exp_id, studied_bias=studied_bias, concept_id=concept_id, patch_id=patch_size_id)
show_freq(exp_name="CMNISTb", exp_id=2, studied_bias=5, concept_id=8, patch_id=6, exp_type="b")
# 1, 4, 8, 6 is good

#### Bias estimator computing
Compute the bias estinator value (or bias value). The "paper" version is the one used in the paper. The "alternative" mode is a second explored version that did not perform well.

In [ ]:
def get_bias_estimator(exp_name, exp_id, exp_type, concept_id, patch_id, backprop_step, bias_threshold, mode="paper"):
    number_of_concept = concept_id
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_parameters = pkl.load(f)
    concept_res = concept_parameters["concept_results"][backprop_step]
    debias_results = {"bias_vectors": [], "bias_threshold": bias_threshold, "all_values": []}

    for label in concept_res.keys():
        if mode == "alternative":
            if "false_n" in concept_res.get(label, {}):
                results_fn = concept_res[label]["false_n"]
                valid_n = results_fn["concept_base"] <= 0
                added_n = ((results_fn["concept_descent"] > 0) & valid_n).sum(axis=0)
                valid_n = valid_n.sum(axis=0)
            else:
                valid_n = 0

            if "false_p" in concept_res.get(label, {}):
                results_fp = concept_res[label]["false_p"]
                results_fp = concept_res[label]["false_p"]
                valid_p = results_fp["concept_base"] > 0
                removed_p = ((results_fp["concept_descent"] <= 0) & valid_p).sum(axis=0)
                valid_p = valid_p.sum(axis=0)
            else:
                valid_p = 0
                removed_p = 0
            
            general_valid = valid_n + valid_p
            general_valid = general_valid + general_valid.sum()/(3*number_of_concept)
            final_res = np.zeros(number_of_concept)

            if isinstance(general_valid, int):
                final_res = 0
            else:
                final_res[general_valid > 0] = (added_n + removed_p)[general_valid > 0] / general_valid[general_valid > 0]
        
        if mode == "paper":
            if "false_n" in concept_res[label]:
                results = concept_res[label]["false_n"]
                appearance = len(results["concept_base"])
                base_freq_n = ((results["concept_base"] > 0).sum(axis=0))/appearance
                descent_freq_n = (results["concept_descent"] > 0).sum(axis=0)/appearance
            else:
                base_freq_n = np.array([0 for i in range(number_of_concept)])
                descent_freq_n = np.array([0 for i in range(number_of_concept)])
            false_n_res = descent_freq_n - base_freq_n
            false_n_res[false_n_res < 0] = 0
            if "false_p" in concept_res[label]:
                results = concept_res[label]["false_p"]
                appearance = len(results["concept_base"])
                base_freq_p = ((results["concept_base"] > 0).sum(axis=0))/appearance
                descent_freq_p = (results["concept_descent"] > 0).sum(axis=0)/appearance
            else: 
                base_freq_p = np.array([0 for i in range(number_of_concept)])
                descent_freq_p = np.array([0 for i in range(number_of_concept)])
            false_p_res = base_freq_p - descent_freq_p
            false_p_res[false_p_res < 0] = 0

            final_res = (false_p_res + false_n_res) /2

        debias_results[label] = final_res
        debias_results["all_values"].append(final_res)
        for rank, value in enumerate(final_res):
            if value > bias_threshold:
                debias_results["bias_vectors"].append((label, rank))
    return debias_results

def get_alignment_curve(exp_name, exp_id, concept_id, patch_id, exp_type, step=1000, mode="paper"):
    """
    Get the alignment between the bias estimation vector from the debias files and the bias alignment vector from get_cmnist_all_bias_alignment in the form of a dataset of bias_estmation and bias_alignment pair to be then drawn on a figure.
    """
    with open(f"{result_folder}models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        align_res = pkl.load(f)["bias_alignment_values"]
    debias_res = get_bias_estimator(exp_name=exp_name, exp_id=exp_id, exp_type=exp_type, concept_id=concept_id, patch_id=patch_id, backprop_step=step, bias_threshold=0.0, mode=mode)["all_values"]
    res_estimators = []
    res_alignments = []
    for label_id in range(len(align_res)):
        res_estimators += list(debias_res[label_id])
        res_alignments += list(align_res[label_id])
    return res_estimators, res_alignments

In [ ]:
# Print the curve of bias estimation vs bias alignment for the first 10 experiments of CMNIST. All concepts are included
concept_id = 8
patch_id = 6
x = []
y = []
for exp_id in range(0, 10):
    estimators, alignment = get_alignment_curve(exp_name="CMNISTb", exp_id=exp_id, concept_id=concept_id, patch_id=patch_id, exp_type="b", mode = "paper")
    x.extend(estimators)
    y.extend(alignment)
plt.plot(x, y, "x")
plt.xlabel("Bias Estimation")
plt.ylabel("Bias Alignment")
# Adding a regression line using numpy.polynomial
z = np.polynomial.Polynomial.fit(x, y, 1)
plt.plot(*z.linspace(), color="red")
# Draw curve of the average alignment of data points above the current value
x_sorted = sorted(x)
y_avg = []
for i in range(len(x_sorted)):
    y_avg.append(np.mean([y[j] for j in range(len(x)) if x[j] >= x_sorted[i]]))
plt.plot(x_sorted, y_avg, color="green")
plt.title("Alignment curve")
plt.show()

#### Bias estimator histogram
Check the distribution of bias estimator depending on bias existence to see if any difference is visible here

In [ ]:
for exp_id in range(10):
    estimators = get_bias_estimator(exp_name="CMNISTb", exp_id=exp_id, exp_type=exp_type, concept_id=concept_id, patch_id=patch_id, backprop_step=step, bias_threshold=0.0, mode="paper")["all_values"]
    for el in estimators:
        bias_estimations_biased.extend(el)
    # with open(f"{result_folder}/models/CMNISTb/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
    #     bias_res = pkl.load(f)["bias_alignment_values"]
    #     for el in bias_res:
    #         bias_estimations_biased.extend(el)
for exp_id in range(10):
        estimators = get_bias_estimator(exp_name="CMNISTu", exp_id=exp_id, exp_type=exp_type, concept_id=concept_id, patch_id=patch_id, backprop_step=step, bias_threshold=0.0, mode="paper")["all_values"]
        for el in estimators:
            bias_estimations_unbiased.extend(el)
    # with open(f"{result_folder}/models/CMNISTu/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
    #     bias_res = pkl.load(f)["bias_alignment_values"]
    # for el in bias_res:
    #     bias_estimations_debiased.extend(el)
plt.hist(bias_estimations_biased, bins=50, alpha=0.5, label="Biased model")
plt.hist(bias_estimations_unbiased, bins=50, alpha=0.5, label="Unbiased model")
plt.xlabel("Bias Estimation")
plt.ylabel("Frequency")
plt.title("Histogram of bias estimation values")
plt.legend()
plt.show()

#### Bias estimator/aligmnment curve
Draw the figures from bias value/alignment value correlation for CMNIST, Appendix C.2

In [ ]:
concept_id = 8
patch_id = 6
for step in [1000, 20000]:
    x = []
    y = []
    for exp_id in range(0, 10):
        estimators, alignments = get_alignment_curve(exp_name="CMNISTb", exp_id=exp_id, concept_id=concept_id, patch_id=patch_id, exp_type="b", step=step, mode = "paper")
        x.extend(estimators)
        y.extend(alignments)
    plt.plot(x, y, "x", label=f"Step {step}")
    # Adding a regression line using numpy.polynomial
    z = np.polynomial.Polynomial.fit(x, y, 1)
    plt.plot(*z.linspace(), label=f"{step}")
    # Draw curve of the average alignment of data points above the current value
    x_sorted = sorted(x)
    y_avg = []
    for i in range(len(x_sorted)):
        y_avg.append(np.mean([y[j] for j in range(len(x)) if x[j] >= x_sorted[i]]))
    plt.plot(x_sorted, y_avg, color="green")
plt.xlabel("Bias Estimation")
plt.ylabel("Bias Alignment")
plt.legend()


plt.xlim(0, 1)
plt.title("Alignment curve")
# plt.legend()
plt.show()


for step in [20000]:
    x = []
    y = []
    for exp_id in range(0, 10):
        estimators, alignments = get_alignment_curve(exp_name="CMNISTu", exp_id=exp_id, concept_id=concept_id, patch_id=patch_id, exp_type="b", step=step, mode = "paper")
        x.extend(estimators)
        y.extend(alignments)
    plt.plot(x, y, "x", label=f"Step {step}")
    # Adding a regression line using numpy.polynomial
    z = np.polynomial.Polynomial.fit(x, y, 1)
    plt.plot(*z.linspace(), label=f"{step}")
    # Draw curve of the average alignment of data points above the current value
    x_sorted = sorted(x)
    y_avg = []
    for i in range(len(x_sorted)):
        y_avg.append(np.mean([y[j] for j in range(len(x)) if x[j] >= x_sorted[i]]))
    plt.plot(x_sorted, y_avg, color="green")
plt.xlabel("Bias Estimation")
plt.ylabel("Bias Alignment")
plt.legend()


plt.xlim(0, 1)
plt.title("Alignment curve")
# plt.legend()
plt.show()

In [ ]:
# Generate a big figure that incorporates a subfigure similar to the one above with as line of the subfigure the bias of the model (biased/unbiased) and as column the bias of the audit (biased/unbiased/differently biased)
fig, axes = plt.subplots(1, 5, figsize=(24,5))
concept_id = 8
patch_id = 6
step = 20000
letters = ["a)", "b)", "c)", "d)", "e)"]
letter_rank=0
for (exp_type, audit_type), ax in zip([("b", "b"), ("b", "u"), ("b", "ob"), ("u", "b"), ("u", "u")], axes):
    x = []
    y = []
    for exp_id in range(0, 10):
        estimators, alignments = get_alignment_curve(exp_name="CMNIST" + exp_type, exp_id=exp_id, concept_id=concept_id, patch_id=patch_id, exp_type=audit_type, step=step)
        x.extend(estimators)
        y.extend(alignments)
    ax.plot(x, y, "x")
    # Adding a regression line using numpy.polynomial
    z = np.polynomial.Polynomial.fit(x, y, 1)
    ax.plot(*z.linspace(), color="red")
    # Draw curve of the average alignment of data points above the current value
    x_sorted = sorted(x)
    y_avg = []
    for i in range(len(x_sorted)):
        y_avg.append(np.mean([y[j] for j in range(len(x)) if x[j] >= x_sorted[i]]))
    ax.plot(x_sorted, y_avg, color="green")
    ax.set_xlabel("Bias Estimation")
    ax.set_ylabel("Bias Alignment")
    ax.set_xlim(0, 1)
    ax.set_ylim(-1, 1)
    if exp_type == "b":
        model_bias = "Biased"
    else:
        model_bias = "Unbiased"
    if audit_type == "b":
        audit_bias = "biased"
    elif audit_type == "u":
        audit_bias = "unbiased"
    else:
        audit_bias = "differently biased"
    ax.set_title(f"{letters[letter_rank]} {model_bias} model, {audit_bias} audit")
    letter_rank+=1
# plt.suptitle("Alignment curves for different bias configurations", fontsize=16)
plt.tight_layout()
plt.show()

### Waterbirds bias estimator alignment
Draw the bias value/bias alignment figures for Waterbirds for part 4.1

In [ ]:
def get_waterbirds_all_bias_alignment(exp_name,  exp_id, concept_id, patch_id, exp_type="b"):
    number_of_concepts = concept_id
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_b_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_res = pkl.load(f)
    res =  []
    for studied_bias in range(2):
        crops_u = concept_res["concept_parameters"][studied_bias]["crops_u"]
        crops = concept_res["concept_parameters"][studied_bias]["crops"]
        masks = concept_res["concept_parameters"][studied_bias]["crop_masks"]
        bias_res = []
        for c_id in range(number_of_concepts):
            best_crops_ids = np.argsort(crops_u[:, c_id])[::-1][:100]
            best_masks = masks[best_crops_ids]
            alignment = 1-(((best_masks > 0).mean()))
            bias_res.append(alignment)
        
        res.append(bias_res)
    return res

In [ ]:
def get_alignment_curve(exp_name, exp_id, concept_id, patch_id, mode="paper"):
    """
    Get the alignment between the bias estimation vector from the debias files and the bias alignment vector from get_waterbirds_all_bias_alignment in the form of a dataset of bias_estmation and bias_alignment pair to be then drawn on a figure.
    """
    align_res = get_waterbirds_all_bias_alignment(exp_name=exp_name, exp_id=exp_id, concept_id=concept_id, patch_id=patch_id)
    # with open(f"{result_folder}/models/{exp_name}/debias_{exp_id}_b{concept_id}_{patch_id}.pkl", "rb") as f:
    #     debias_res = pkl.load(f)["all_values"]
    debias_res = get_bias_estimator(exp_name=exp_name, exp_id=exp_id, exp_type="b", concept_id=concept_id, patch_id=patch_id, backprop_step=20000, bias_threshold=0.0, mode=mode)
    res = []
    for label_id in range(len(align_res)):
        for bias_id in range(len(align_res[label_id])):
            res.append((debias_res[label_id][bias_id], np.abs(align_res[label_id][bias_id])))
        # bias_id = debias_res[label_id].argmax()
        # res.append((debias_res[label_id][bias_id], np.abs(align_res[label_id][bias_id])))
    return res

# Print the curve of bias estimation vs bias alignment for the first 10 experiments of CMNIST. All concepts are included
concept_id = 10
patch_id = 50
alignment_curve = []
for exp_id in range(0, 10):
    alignment_curve.extend(get_alignment_curve(exp_name="Waterbirds18", exp_id=exp_id, concept_id=concept_id, patch_id=patch_id, mode="paper"))
x=[el[0] for el in alignment_curve]
y=[el[1] for el in alignment_curve]
plt.plot(x, y, "x")
plt.xlabel("Bias Estimation")
plt.ylabel("Bias Alignment")
# Adding a regression line using numpy.polynomial
z = np.polynomial.Polynomial.fit(x, y, 1)
plt.plot(*z.linspace(), color="red")
# Draw curve of the average alignment of data points above the current value
x_sorted = sorted(x)
y_avg = []
for i in range(len(x_sorted)):
    y_avg.append(np.mean([y[j] for j in range(len(x)) if x[j] >= x_sorted[i]]))
plt.plot(x_sorted, y_avg, color="green")
plt.title("Alignment curve")
plt.show()

### Correlation
Correlation with the ground-truth bias labels, part 4.3 and C.3

In [ ]:
from scipy.stats import mannwhitneyu, ttest_ind, shapiro
def bias_correlations(exp_name, exp_range, concept_id, patch_id, exp_type="b"):
    res = {"bias":[], "random": []}
    for exp_id in exp_range:
        with open(f"{result_folder}/models/" + exp_name + "/debiasing_" + str(exp_id) + f"_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
            debias_res = pkl.load(f)
        bias_concepts = debias_res["bias_estimator"]["bias_vectors"]
        for label in debias_res["chi2_test"]["merged"]:
            for concept in range(concept_id):
                cluster_label = debias_res["cluster_labels"][concept_id * label + concept]
                concept_res = [debias_res["chi2_test"]["merged"][label]["chi2_stats"][cluster_label],
                        debias_res["chi2_test"]["merged"][label]["p_values"][cluster_label],
                        debias_res["chi2_test"]["merged"][label]["mcc_values"][cluster_label]]
                if (label, concept) in bias_concepts:
                    res["bias"].append(concept_res)
                else:
                    res["random"].append(concept_res)
    return res

def test_bias_correlation(exp_name, exp_range, concept_id, patch_id, exp_type="b"):
    res = bias_correlations(exp_name, exp_range, concept_id, patch_id, exp_type)
    bias_values = np.array([el[2] for el in res["bias"] if el[1] < 0.05])
    bias_values = bias_values[~np.isnan(bias_values)]
    random_values = np.array([el[2] for el in res["random"] if el[1] < 0.05])
    random_values = random_values[~np.isnan(random_values)]
    bias_values = np.abs(bias_values)
    random_values = np.abs(random_values)
    print(len(bias_values), len(random_values))
    t_stat, p_value = mannwhitneyu(bias_values, random_values, alternative="greater")

    print(f"T-statistic: {t_stat}, P-value: {p_value}, Proportion: {len(bias_values)/len(np.array([el[2] for el in res['bias']])), len(random_values)/len(np.array([el[2] for el in res['random']]))}, Mean and std bias: {round(bias_values.mean(), 3)} $\pm$ {round(bias_values.std(), 3)}, Mean and std random: {round(random_values.mean(), 3)} $\pm$ {round(random_values.std(), 3)}")
    return t_stat, p_value


print("\nBiased CMNIST, biased audit:")
test_bias_correlation(exp_name="CMNISTb", exp_range=range(10), concept_id=8, patch_id=6, exp_type="b")

print("\nBiased CMNIST, unbiased audit:")
test_bias_correlation(exp_name="CMNISTb", exp_range=range(10), concept_id=8, patch_id=6, exp_type="u")

print("\nBiased CMNIST, differently biased audit:")
test_bias_correlation(exp_name="CMNISTb", exp_range=range(10), concept_id=8, patch_id=6, exp_type="ob")

print("\nUnbiased CMNIST, biased audit:")
test_bias_correlation(exp_name="CMNISTu", exp_range=range(10), concept_id=8, patch_id=6, exp_type="b")  

print("\nUnbiased CMNIST, unbiased audit:")
test_bias_correlation(exp_name="CMNISTu", exp_range=range(10), concept_id=8, patch_id=6, exp_type="u")

print("\nWaterbirds:")
a, b = test_bias_correlation(exp_name="Waterbirds18", exp_range=range(10), concept_id=10, patch_id=50, exp_type="b")

print("\nCelebA:")
a, b = test_bias_correlation(exp_name="CelebA", exp_range=range(10), concept_id=10, patch_id=50, exp_type="b")

In [ ]:
def get_bias_concept_proportion(exp_name, exp_range, concept_id, patch_id, exp_type="b"):
    proportions = []
    for exp_id in exp_range:
        with open(f"{result_folder}/models/" + exp_name + "/debiasing_" + str(exp_id) + f"_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
            debias_res = pkl.load(f)
        proportions.append(len(debias_res["bias_estimator"]["bias_vectors"]))
    print(f"Average number of bias concepts identified: {np.mean(proportions)}, std: {np.std(proportions)}")
    return proportions
print("Biased CMNIST, biased audit:")
get_bias_concept_proportion(exp_name="CMNISTb", exp_range=range(10), concept_id=8, patch_id=6, exp_type="b")
print("Biased CMNIST, unbiased audit:")
get_bias_concept_proportion(exp_name="CMNISTb", exp_range=range(10), concept_id=8, patch_id=6, exp_type="u")
print("Biased CMNIST, differently biased audit:")
get_bias_concept_proportion(exp_name="CMNISTb", exp_range=range(10), concept_id=8, patch_id=6, exp_type="ob")
print("Unbiased CMNIST, biased audit:")
get_bias_concept_proportion(exp_name="CMNISTu", exp_range=range(10), concept_id=8, patch_id=6, exp_type="b")  
print("Unbiased CMNIST, unbiased audit:")
get_bias_concept_proportion(exp_name="CMNISTu", exp_range=range(10), concept_id=8, patch_id=6, exp_type="u")
print("Waterbirds:")
get_bias_concept_proportion(exp_name="Waterbirds18", exp_range=range(10), concept_id=10, patch_id=50, exp_type="b")
print("CelebA:")
get_bias_concept_proportion(exp_name="CelebA", exp_range=range(10), concept_id=10, patch_id=50, exp_type="b")

In [ ]:
# Draw bias estimator vs bias correlation coefficient mcc for all experience of CelebA
exp_range = range(10)
concept_id = 10
patch_id = 50
bias_estimators = []
mcc_values = []
for exp_id in exp_range:
    with open(f"{result_folder}/models/Waterbirds18/debiasing_{exp_id}_b_{concept_id}_{patch_id}.pkl", "rb") as f:
        debias_res = pkl.load(f)
    for label in debias_res["chi2_test"]["merged"]:
        for concept in range(concept_id):
            cluster_label = debias_res["cluster_labels"][concept_id * label + concept]
            concept_res = [debias_res["chi2_test"]["merged"][label]["chi2_stats"][cluster_label],
                    debias_res["chi2_test"]["merged"][label]["p_values"][cluster_label],
                    debias_res["chi2_test"]["merged"][label]["mcc_values"][cluster_label]]
            bias_estimators.append(debias_res["bias_estimator"]["all_values"][label][concept])
            mcc_values.append(concept_res[2])
plt.scatter(bias_estimators, mcc_values)
plt.xlabel("Bias Estimation")
plt.ylabel("MCC value")
plt.title("Bias Estimation vs MCC value for CelebA")
# With a regression line
try:
    z = np.polyfit(bias_estimators, mcc_values, 1)
    p = np.poly1d(z)
    plt.plot(bias_estimators, p(bias_estimators), color="red")
except:
    pass
plt.show()

In [ ]:
# Draw bias estimator vs bias correlation coefficient mcc for all experience of CelebA
exp_range = range(10)
concept_id = 10
patch_id = 50
mcc_values_celeb = []
mcc_values_water = []
for exp_id in exp_range:
    with open(f"{result_folder}/models/CelebA/debiasing_{exp_id}_b_{concept_id}_{patch_id}.pkl", "rb") as f:
        debias_res = pkl.load(f)
    for label in debias_res["chi2_test"]["merged"]:
        for concept in range(concept_id):
            cluster_label = debias_res["cluster_labels"][concept_id * label + concept]
            concept_res = [debias_res["chi2_test"]["merged"][label]["chi2_stats"][cluster_label],
                    debias_res["chi2_test"]["merged"][label]["p_values"][cluster_label],
                    debias_res["chi2_test"]["merged"][label]["mcc_values"][cluster_label]]
            mcc_values_celeb.append(concept_res[2])
for exp_id in exp_range:
    with open(f"{result_folder}/models/Waterbirds18/debiasing_{exp_id}_b_{concept_id}_{patch_id}.pkl", "rb") as f:
        debias_res = pkl.load(f)
    for label in debias_res["chi2_test"]["merged"]:
        for concept in range(concept_id):
            cluster_label = debias_res["cluster_labels"][concept_id * label + concept]
            concept_res = [debias_res["chi2_test"]["merged"][label]["chi2_stats"][cluster_label],
                    debias_res["chi2_test"]["merged"][label]["p_values"][cluster_label],
                    debias_res["chi2_test"]["merged"][label]["mcc_values"][cluster_label]]
            mcc_values_water.append(concept_res[2])
plt.hist(mcc_values_water, bins=20, alpha=0.5, range=(-1, 1), label="Waterbirds")
plt.hist(mcc_values_celeb, bins=20, alpha=0.5, range=(-1, 1), label="CelebA")
plt.xlabel("MCC value")
plt.ylabel("Frequency")
plt.title("Distribution of MCC values for CelebA and Waterbirds")
plt.legend()
plt.show()

### Experiment observation
Allows to see the entire identification pipeline for a model

In [ ]:
def show(img, im_min=None, im_max=None, **kwargs):
  img = np.array(img)
  if img.shape[0] == 3:
    img = img.transpose(1, 2, 0)
  if im_min is None:
    im_min = img.min()
  if im_max is None:    im_max = img.max()
  img -= im_min
  img /= im_max - im_min

  # img -= img.min();img /= img.max()

  plt.imshow(img, **kwargs); plt.axis('off')
  return img

def mask_show(img, **kwargs):
  img = np.array(img)
  if img.shape[0] == 3:
    img = img.transpose(1, 2, 0)

  img -= img.min()
  plt.imshow(img, **kwargs); plt.axis('off')

def show_bias_crops(concept_parameters, label, concept, amount=5, column=False):
  crops_u = concept_parameters[label]["crops_u"]
  crops = concept_parameters[label]["crops"]
  if "crop_masks" in concept_parameters[label]:
    masks = concept_parameters[label]["crop_masks"]
  best_crops_ids = np.argsort(crops_u[:, concept])[::-1][:amount]
  best_crops = crops[best_crops_ids]
  im_min, im_max = crops.min(), crops.max()
  if "crop_masks" in concept_parameters[label]:
    best_masks = masks[best_crops_ids]
  imgs = []
  crop_w, crop_h = 3, 3  # constant size per crop (inches)
  plt.figure(figsize=(crop_w * amount, crop_h))
  for i in range(amount):
    if column:
       plt.subplot(amount, 1, i + 1)
    else:
      plt.subplot(1, amount, i + 1)
    imgs.append(show(best_crops[i], im_min=im_min, im_max=im_max))
  plt.tight_layout()
  plt.show()

  if "crop_masks" in concept_parameters[label]:
    plt.figure(figsize=(crop_w * amount, crop_h))
    for i in range(amount):
      plt.subplot(1, amount, i + 1)
      mask_show(best_masks[i])
    plt.tight_layout()
    plt.show()
  print('\n\n')
  return imgs

In [ ]:
exp_name = "CMNISTb"
concept_id = 8
patch_id = 6
exp_id = 4
exp_type = "u"
with open(f"{result_folder}/models/{exp_name}/model_{exp_id}.pkl", "rb") as f:
    model = pkl.load(f)
with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
    concept_res = pkl.load(f)
with open(f"{result_folder}/models/{exp_name}/debiasing_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
    debias_res = pkl.load(f)
print(f"Base accuracy: {model['correctness_matrix'].sum()/model['appearance_matrix'].sum()}, Debiased accuracy: {debias_res['debiasing_impact'][0].sum()/debias_res['debiasing_impact'][1].sum()}")
fig, axes = plt.subplots(1, 2, figsize=(15,5))
sb.heatmap(model["correctness_matrix"]/model["appearance_matrix"], annot=True, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("Correctness matrix")
sb.heatmap(debias_res["debiasing_impact"][0]/debias_res["debiasing_impact"][1] - model["correctness_matrix"]/model["appearance_matrix"], annot=True, ax=axes[1], center=0, vmin=-0.1, vmax=0.1)
axes[1].set_title("Debiasing impact")
plt.show()
for el in debias_res["bias_estimator"]["bias_vectors"]:
    print(f"Label {el[0]}, concept {el[1]}, chi2 stat: {debias_res['chi2_test']['merged'][el[0]]['chi2_stats'][debias_res['cluster_labels'][concept_id * el[0] + el[1]]]}, p-value: {debias_res['chi2_test']['merged'][el[0]]['p_values'][debias_res['cluster_labels'][concept_id * el[0] + el[1]]]}, mcc value: {debias_res['chi2_test']['merged'][el[0]]['mcc_values'][debias_res['cluster_labels'][concept_id * el[0] + el[1]]]}, bias estimation: {debias_res['bias_estimator']['all_values'][el[0]][el[1]]}")
    show_bias_crops(concept_parameters=concept_res["concept_parameters"], label=el[0], concept=el[1], amount=14)

## Debiasing Comparisons
Comparing bias mitigation experiments, part 4.4 and C.4

In [ ]:
def compare_debiasing(exp_name, exp_type, exp_id_range, concept_id, patch_id, result_folder):
    res = {key : {"Accuracy":[], "Worse class acc":[], "Worse group acc":[]} for key in ["Normal", "Debias", "Ablation"]}
    for exp_id in exp_id_range:
        with open(f"{result_folder}/models/" + exp_name + "/model_" + str(exp_id) + ".pkl", "rb") as f:
            model_res = pkl.load(f)
        # stat = model_res["correctness_matrix"]/model_res["appearance_matrix"]
        # stat = np.concatenate((stat, np.atleast_2d(stat.mean(axis=1)).T), axis =1)
        # stat = np.concatenate((stat, np.atleast_2d(stat.mean(axis=0))), axis =0)
        # res["Normal"]["AC-score"].append(compute_ac_score(model_res["correctness_matrix"], model_res["appearance_matrix"]))
        res["Normal"]["Accuracy"].append(model_res["correctness_matrix"].sum()/model_res["appearance_matrix"].sum())
        res["Normal"]["Worse class acc"].append((model_res["correctness_matrix"].sum(axis=1)/model_res["appearance_matrix"].sum(axis=1)).min())
        res["Normal"]["Worse group acc"].append((model_res["correctness_matrix"]/model_res["appearance_matrix"]).min())

        with open(f"{result_folder}/models/" + exp_name + "/debiasing_" + str(exp_id) + f"_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
            model_res = pkl.load(f)
        if "debiasing_impact" in model_res:
            debias_res = {"correctness_matrix": model_res["debiasing_impact"][0], "appearance_matrix": model_res["debiasing_impact"][1]}
        else:
            continue
        # stat = model_res["correctness_matrix"]/model_res["appearance_matrix"]
        # stat = np.concatenate((stat, np.atleast_2d(stat.mean(axis=1)).T), axis =1)
        # stat = np.concatenate((stat, np.atleast_2d(stat.mean(axis=0))), axis =0)
        # res["Debias"]["AC-score"].append(compute_ac_score(debias_res["correctness_matrix"], debias_res["appearance_matrix"]))
        res["Debias"]["Accuracy"].append(debias_res["correctness_matrix"].sum()/debias_res["appearance_matrix"].sum())
        res["Debias"]["Worse class acc"].append((debias_res["correctness_matrix"].sum(axis=1)/debias_res["appearance_matrix"].sum(axis=1)).min())
        res["Debias"]["Worse group acc"].append((debias_res["correctness_matrix"]/debias_res["appearance_matrix"]).min())

        if "ablation_results" in model_res:
            # print("ablation found")
            ablation_res = model_res["ablation_results"]
            # print(ablation_res)
            for bias_labels, (correctness, appearance) in ablation_res:
                # stat = correctness/appearance
                # stat = np.concatenate((stat, np.atleast_2d(stat.mean(axis=1)).T), axis =1)
                # stat = np.concatenate((stat, np.atleast_2d(stat.mean(axis=0))), axis =0)
                res["Ablation"]["Accuracy"].append(correctness.sum()/appearance.sum())
                res["Ablation"]["Worse class acc"].append((correctness.sum(axis=1)/appearance.sum(axis=1)).min())
                res["Ablation"]["Worse group acc"].append((correctness/appearance).min())

    for exp_type in ["Normal", "Debias", "Ablation"]:
        print()
        for metric in ["Accuracy", "Worse class acc", "Worse group acc"]:
            arr = np.array(res[exp_type][metric])
            print(f"{round(arr.mean()*100, 1)} +- {round(100 * arr.std(), 1)}", end=" & ")
    return res

print("CMNIST biased model, biased audit:")
compare_debiasing(exp_name="CMNISTb", exp_type="b", exp_id_range=range(10), concept_id=8, patch_id=6, result_folder=result_folder)
print("\n")
print("CMNIST biased model, unbiased audit:")
compare_debiasing(exp_name="CMNISTb", exp_type="u", exp_id_range=range(10), concept_id=8, patch_id=6, result_folder=result_folder)
print("\n")
print("CMNIST biased model, differently biased audit:")
compare_debiasing(exp_name="CMNISTb", exp_type="ob", exp_id_range=range(10), concept_id=8, patch_id=6, result_folder=result_folder)
print("\n")
print("CMNIST unbiased model, biased audit:")
compare_debiasing(exp_name="CMNISTu", exp_type="b", exp_id_range=range(10), concept_id=8, patch_id=6, result_folder=result_folder)
print("\n")
print("CMNIST unbiased model, unbiased audit:")
compare_debiasing(exp_name="CMNISTu", exp_type="u", exp_id_range=range(10), concept_id=8, patch_id=6, result_folder=result_folder)
print("\n")
print("Waterbirds:")
compare_debiasing(exp_name="Waterbirds18", exp_type="b", exp_id_range=range(10), concept_id=10, patch_id=50, result_folder=result_folder)
print("\n")
print("CelebA:")
compare_debiasing(exp_name="CelebA", exp_type="b", exp_id_range=range(10), concept_id=10, patch_id=50, result_folder=result_folder)
print(" ")

### Amplification effect
Studying the effect of amplifying a specific concept. Did not make it into the final paper

In [ ]:
with open(f"{result_folder}/models/CMNISTb/debiasing_3_u_8_6.pkl", "rb") as f:
    model_res = pkl.load(f)
fig, axes = plt.subplots(1, 3, figsize=(25,5))
sb.heatmap(model_res["bias_amplification"][0], ax=axes[0], vmin=0, vmax=4000, annot=True, fmt="g")
axes[0].set_xlabel("True label")
axes[0].set_ylabel("Predicted label")
sb.heatmap(model_res["bias_amplification"][1], ax=axes[1], vmin=0, vmax=4000, annot=True, fmt="g")
axes[1].set_xlabel("True label")
axes[1].set_ylabel("Predicted label")
# Plot the difference between both heatmaps
sb.heatmap(np.array(model_res["bias_amplification"][1]) - np.array(model_res["bias_amplification"][0]), ax=axes[2], center=0, vmin=-50, vmax=50, annot=True, fmt="g")
axes[2].set_xlabel("True label")
axes[2].set_ylabel("Predicted label")
merged_bias_label = model_res["bias_merged"][0]
gen_label = list(model_res["cluster_labels"]).index(merged_bias_label)
bias_label = (gen_label//8, gen_label%8)
print(bias_label)
for res in model_res["bias_amplification"]:
    print("Accuracy: ", res.trace()/res.sum())

In [ ]:
with open(f"{result_folder}/models/Waterbirds18/debiasing_9_b_10_50.pkl", "rb") as f:
    model_res = pkl.load(f)
fig, axes = plt.subplots(1, 3, figsize=(25,5))
sb.heatmap(model_res["bias_amplification"][0], ax=axes[0], vmin=0, vmax=4000, annot=True, fmt="g")
axes[0].set_xlabel("True label")
axes[0].set_ylabel("Predicted label")
sb.heatmap(model_res["bias_amplification"][1], ax=axes[1], vmin=0, vmax=4000, annot=True, fmt="g")
axes[1].set_xlabel("True label")
axes[1].set_ylabel("Predicted label")
# Plot the difference between both heatmaps
sb.heatmap(np.array(model_res["bias_amplification"][1]) - np.array(model_res["bias_amplification"][0]), ax=axes[2], center=0, vmin=-50, vmax=50, annot=True, fmt="g")
axes[2].set_xlabel("True label")
axes[2].set_ylabel("Predicted label")
merged_bias_label = model_res["bias_merged"][0]
gen_label = list(model_res["cluster_labels"]).index(merged_bias_label)
bias_label = (gen_label//10, gen_label%10)
print(bias_label)
for res in model_res["bias_amplification"]:
    print("Accuracy: ", res.trace()/res.sum())

In [ ]:
def get_proportions_amplified(exp_range, exp_name, exp_type, concept_id, patch_id, result_folder):
    proportions = {"Base": {"label": [], "not_label": []}, "Amplified": {"label": [], "not_label": []}}
    for exp_id in exp_range:
        with open(f"{result_folder}/models/" + exp_name + "/debiasing_" + str(exp_id) + f"_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
            debias_res = pkl.load(f)
        if len(debias_res["bias_merged"]) == 0:
            continue
        merged_bias_label = debias_res["bias_merged"][0]
        gen_label = list(debias_res["cluster_labels"]).index(merged_bias_label)
        bias_label = (gen_label//concept_id, gen_label%concept_id)
        label = bias_label[0]

        not_label = np.array([i for i in range(len(debias_res["bias_amplification"][0])) if i != label])
        proportions["Base"]["label"].append(debias_res["bias_amplification"][0][label][label]/debias_res["bias_amplification"][0][:, label].sum())
        proportions["Base"]["not_label"].append(debias_res["bias_amplification"][0][label][not_label].sum()/debias_res["bias_amplification"][0][:,not_label].sum())
        proportions["Amplified"]["label"].append(debias_res["bias_amplification"][1][label][label]/debias_res["bias_amplification"][1][:, label].sum())
        proportions["Amplified"]["not_label"].append(debias_res["bias_amplification"][1][label][not_label].sum()/debias_res["bias_amplification"][1][:,not_label].sum())
    # print(f"Proportion of correcly assigned to {label}: {round(np.mean(proportions['Base']['label']), 3)}, missclassified as {label}: {round(np.mean(proportions['Base']['not_label']), 3)}, correctly assigned to {label} after amplification: {round(np.mean(proportions['Amplified']['label']), 3)}, misclassified as {label} after amplification: {round(np.mean(proportions['Amplified']['not_label']), 3)}")
    return np.array(proportions["Amplified"]["label"]) - np.array(proportions["Base"]["label"]), np.array(proportions["Amplified"]["not_label"]) - np.array(proportions["Base"]["not_label"])

print("CMNISTbb:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="CMNISTb", exp_type="b", concept_id=8, patch_id=6, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))
print("CMNISTbu:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="CMNISTb", exp_type="u", concept_id=8, patch_id=6, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))
print("CMNISTbob:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="CMNISTb", exp_type="ob", concept_id=8, patch_id=6, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))
print("CMNISTub:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="CMNISTu", exp_type="b", concept_id=8, patch_id=6, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))
print("CMNISTuu:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="CMNISTu", exp_type="u", concept_id=8, patch_id=6, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))
print("Waterbirds:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="Waterbirds18", exp_type="b", concept_id=10, patch_id=50, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))
print("CelebA:")
for el in get_proportions_amplified(exp_range=range(10), exp_name="CelebA", exp_type="b", concept_id=10, patch_id=50, result_folder=result_folder):
    # print(el)
    print(round(el.mean()*100, 1), round(el.std()*100, 1))

## Hypothesis figure
Drawing of figure 4 from the article

In [ ]:
# Script that recover for a certain experiment and label the most and least bias aligned concepts (using get_cmnist_all_bias_alignment) and then map the concept activation for these concepts (least as x, most as y) for the false_negatives and false_positives samples. The zero would be normalized as a single point in the negative part of the quadrant.
def get_concept_activation_for_most_least_aligned_concepts(exp_name, exp_id, concept_id, patch_id, exp_type="b", label=0, backprop_step=1000, alignment_threshold=0.85):
    align_res = get_waterbirds_all_bias_alignment(exp_name=exp_name, exp_id=exp_id, concept_id=concept_id, patch_id=patch_id, exp_type=exp_type)
    concept_parameters = None
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_parameters = pkl.load(f)
    concept_res = concept_parameters["concept_results"][backprop_step]
    res = {"false_n": None, "false_p": None}
    bias_alignments = align_res[label]
    print("Bias alignments: ", bias_alignments)
    # most_aligned_concepts = np.array(bias_alignments) >= alignment_threshold
    # least_aligned_concepts = ~most_aligned_concepts
    most_aligned_concepts = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=bool)
    least_aligned_concepts = np.array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0], dtype=bool)
    if "false_n" in concept_res[label]:
        results_fn = concept_res[label]["false_n"]
        false_n_activations_base_b = results_fn["concept_base"][:, most_aligned_concepts].mean(axis=1)
        false_n_activations_base_u = results_fn["concept_base"][:, least_aligned_concepts].mean(axis=1)
        false_n_activations_descent_b = results_fn["concept_descent"][:, most_aligned_concepts].mean(axis=1)
        false_n_activations_descent_u = results_fn["concept_descent"][:, least_aligned_concepts].mean(axis=1)
        false_n_activations_base = np.concatenate((false_n_activations_base_u[:, None], false_n_activations_base_b[:, None]), axis=1)
        false_n_activations_descent = np.concatenate((false_n_activations_descent_u[:, None], false_n_activations_descent_b[:, None]), axis=1)
        res["false_n"] = (false_n_activations_base, false_n_activations_descent)
    if "false_p" in concept_res[label]:
        results_fp = concept_res[label]["false_p"]
        false_p_activations_base_b = results_fp["concept_base"][:, most_aligned_concepts].mean(axis=1)
        false_p_activations_base_u = results_fp["concept_base"][:, least_aligned_concepts].mean(axis=1)
        false_p_activations_descent_b = results_fp["concept_descent"][:, most_aligned_concepts].mean(axis=1)
        false_p_activations_descent_u = results_fp["concept_descent"][:, least_aligned_concepts].mean(axis=1)
        false_p_activations_base = np.concatenate((false_p_activations_base_u[:, None], false_p_activations_base_b[:, None]), axis=1)
        false_p_activations_descent = np.concatenate((false_p_activations_descent_u[:, None], false_p_activations_descent_b[:, None]), axis=1)
        res["false_p"] = (false_p_activations_base, false_p_activations_descent)
    return res
concept_id = 10
patch_id = 50
exp_id = 1
exp_type = "b"
label=1
res = get_concept_activation_for_most_least_aligned_concepts(exp_name="Waterbirds18", exp_id=exp_id, label=label, concept_id=concept_id, patch_id=patch_id, exp_type=exp_type, backprop_step=1000)
false_n_base = res["false_n"][0]
false_n_descent = res["false_n"][1]
false_p_base = res["false_p"][0][:len(false_n_base)]
false_p_descent = res["false_p"][1][:len(false_n_base)]
plt.figure(figsize=(12, 5))
# All in the same plot but different color and draw a line betwwen a sample and it's descent version
plt.subplot(1, 2, 1)
plt.title("a) False Negatives", fontsize=17)
plt.xlabel("Activation on concept 0 (relevant)", color="#009900")
plt.ylabel("Activation on concept 1 (bias-aligned)", color="#FF8000")
# false_n_base[false_n_base < 0.05] = 0
# false_n_descent[false_n_descent <0.05] = 0
# false_p_base[false_p_base < 0.05] = 0
# false_p_descent[false_p_descent < 0.05] = 0
# false_n_base[false_n_base == 0] = -0.2
# false_n_descent[false_n_descent == 0] = -0.2
# false_p_base[false_p_base == 0] = -0.2
# false_p_descent[false_p_descent == 0] = -0.2
for i in range(len(false_n_base)):
    # plt.plot([false_n_base[i, 0], false_n_descent[i, 0]], [false_n_base[i, 1], false_n_descent[i, 1]], color="gray", alpha=0.5)
    arrow_width = 0.0002
    plt.arrow(
        x=false_n_base[i, 0], 
        y=false_n_base[i, 1], 
        dx=(false_n_descent[i, 0]-false_n_base[i, 0]), 
        dy=(false_n_descent[i, 1]-false_n_base[i, 1]), 
        width=arrow_width, 
        head_width=8*arrow_width,
        head_length=12*arrow_width,
        length_includes_head=True,
        color="grey", 
        alpha=0.4
        )
plt.scatter(false_n_base[:, 0], false_n_base[:, 1], marker="x", color="blue", label="Base")
plt.scatter(false_n_descent[:, 0], false_n_descent[:, 1], marker="x", color="red", label="After descent")
# Draw axis for 0
plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.axvline(0, color="black", linestyle="--", alpha=0.5)

plt.legend()
# Same for false positives
plt.subplot(1, 2, 2)
plt.title("b) False Positives", fontsize=17)
plt.xlabel("Activation on concept 0 (relevant)", color="#009900", fontsize=12)
# plt.ylabel("Activation on concept 1 (bias-aligned)", color="#FF8000")
for i in range(len(false_p_base )):
    # plt.plot([false_p_base[i, 0], false_p_descent[i, 0]], [false_p_base[i, 1], false_p_descent[i, 1]], color="gray", alpha=0.5)
    arrow_width = 0.0003
    plt.arrow(
        x=false_p_base[i, 0], 
        y=false_p_base[i, 1], 
        dx=(false_p_descent[i, 0]-false_p_base[i, 0]), 
        dy=(false_p_descent[i, 1]-false_p_base[i, 1]), 
        width=arrow_width, 
        head_width=8*arrow_width,
        head_length=12*arrow_width,
        length_includes_head=True,
        color="grey", 
        alpha=0.4
        )
plt.scatter(false_p_base[:, 0], false_p_base[:, 1], marker="x", color="blue", label="Base")
plt.scatter(false_p_descent[:, 0], false_p_descent[:, 1], marker="x", color="red", label="After descent")
plt.legend()
# Draw axis for 0
plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.axvline(0, color="black", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
with open(f"{result_folder}/models/Waterbirds18/concepts_1_b_10_50.pkl", "rb") as f:
    concept_parameters = pkl.load(f)
for i in range(10):
    print(i)
    show_bias_crops(concept_parameters=concept_parameters["concept_parameters"], label=1, concept=i, amount=3, column=True)